# UERC26 — TinyViT-11M Pruning Pipeline (A100 80GB Colab Optimised)

## Hardware target
- **GPU**: A100 80 GB (Google Colab)
- **Training images**: ~180 k
- **Student**: TinyViT-11M (~21 M params)
- **Teacher**: SwinV2-Base (~88 M params, frozen)

## How to run

### One-time setup — run once per Kaggle session
| Cell | Description |
|------|-------------|
| **Cell 1** | Install deps, mount/unzip dataset |
| **Cell 2** | Import modules |
| **Cell 3** | CLAHE & sharpening utilities |
| **Cell 4** | Dataset classes |
| **Cell 5** | Teacher model (SwinV2-Base) + AdaFace head |
| **Cell 6** | Evaluation function & metrics |
| **Cell 7** | Build dataloaders (A100-tuned settings) |
| **Cell 8** | TinyViT-11M student model definition |
| **Cell 9** | Load trained student + teacher; run **baseline eval once** |
| **Cell 10** | Install `torch-pruning` |

### Pruning loop — re-run Cells 11–16 for each pruning attempt
| Cell | Description |
|------|-------------|
| **Cell 11** | Set `PRUNING_RATIO`; configure Taylor pruner |
| **Cell 12** | Accumulate Taylor scores → apply pruning masks |
| **Cell 13** | Re-distillation fine-tuning — saves `best_pruned_tinyvit.pt` |
| **Cell 14** | Compare pruned vs baseline (⚠️ flags for metric floors) |
| **Cell 15** | Full test-set evaluation on pruned model |
| **Cell 16** | Save best pruned model to Google Drive |

> **Iteration tip**: start at `PRUNING_RATIO=0.25`. If Cell 14 shows all ✅ and
> param reduction is not enough, raise to `0.30` or `0.35` and re-run Cells 11–16.
> If ⚠️ appears, lower the ratio.

### Submission — run once after pruning is finalised
| Cell | Description |
|------|-------------|
| **Cell 17** | Extract sequestered features → save to Google Drive |
| **Cell 18** | Final evaluation on test set |

In [ ]:
# Cell 1 — Mount Google Drive, Install Dependencies & Prepare Dataset
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys, os
subprocess.run([sys.executable, '-m', 'pip', 'install', 'timm', 'torch-pruning', '-q'], check=True)

os.makedirs('/content/checkpoints', exist_ok=True)

# ── Update this path to your dataset ZIP in Drive ────────────────
INPUT_ZIP  = '/content/drive/MyDrive/UERC26_modified_new.zip'
EXTRACT_TO = '/content'
DATA_CHECK = '/content/UERC26_modified_new/data/public/dataset_split.csv'

if not os.path.exists(DATA_CHECK):
    print('Unzipping dataset...')
    os.system(f'unzip -q "{INPUT_ZIP}" -d "{EXTRACT_TO}"')
    print('Done.')
else:
    print('Dataset already extracted.')



Mounted at /content/drive
Unzipping dataset...
Done.


In [ ]:
# Cell 2 — Import Modules
import os, csv, time, random, copy, shutil
import numpy as np
from PIL import Image
from pathlib import Path
import cv2
cv2.setNumThreads(0)  # prevent OpenCV from stealing dataloader threads


import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.transforms.v2 as v2
import torchvision.models as models
import timm
import torch_pruning as tp
from tqdm.notebook import tqdm
from scipy.interpolate import interp1d
from sklearn.metrics import roc_curve, roc_auc_score

# Avoid /dev/shm exhaustion on Colab — use filesystem-based tensor sharing
import torch.multiprocessing
torch.multiprocessing.set_sharing_strategy('file_system')

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print('All modules imported.')
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.version.cuda}')
print(f'GPU      : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f'VRAM     : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB' if torch.cuda.is_available() else '')



All modules imported.
PyTorch  : 2.10.0+cu128
CUDA     : 12.8
GPU      : NVIDIA A100-SXM4-80GB
VRAM     : 85.1 GB


In [ ]:
# Cell 3 — CPU Preprocessing Utilities (CLAHE & Sharpening)

def apply_clahe(image_np, clip_limit=2.0, tile_grid_size=(8, 8)):
    lab = cv2.cvtColor(image_np, cv2.COLOR_RGB2LAB)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    lab[:, :, 0] = clahe.apply(lab[:, :, 0])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

def apply_sharpening(image_np, sigma=1.0, amount=0.5):
    blurred = cv2.GaussianBlur(image_np, (0, 0), sigma)
    sharpened = cv2.addWeighted(image_np, 1.0 + amount, blurred, -amount, 0)
    return np.clip(sharpened, 0, 255).astype(np.uint8)

print('Preprocessing utilities defined.')



Preprocessing utilities defined.


In [ ]:
# Cell 4 — Dataset Classes (optimised for A100 + 8-core Colab Pro+)

def _np_to_tensor(img_np):
    """Convert HWC uint8 numpy array to CHW float32 tensor WITHOUT PIL overhead."""
    return torch.from_numpy(img_np.transpose(2, 0, 1).copy()).float().div_(255.0)


class EarIdentificationDataset(Dataset):
    """Lazy-loading WebP training dataset with CPU augmentations."""
    def __init__(self, split, root_dir, split_csv='dataset_split.csv',
                 classes_mapping=None, is_train=True):
        self.root_dir  = root_dir
        self.is_train  = is_train
        self.images    = []

        with open(os.path.join(root_dir, split_csv), 'r') as f:
            for row in csv.reader(f, delimiter='\t'):
                if len(row) >= 2 and row[1] == split:
                    self.images.append(row[0].rsplit('.', 1)[0] + '.webp')

        subjects = sorted(set(os.path.dirname(img) for img in self.images))
        self.classes_mapping = classes_mapping or {s: i for i, s in enumerate(subjects)}
        self.num_classes = len(self.classes_mapping)
        self.images = [img for img in self.images if os.path.dirname(img) in self.classes_mapping]
        self.labels = [self.classes_mapping[os.path.dirname(img)] for img in self.images]

    def __len__(self): return len(self.images)

    def __getitem__(self, idx):
        img_path = os.path.join(self.root_dir, self.images[idx])
        image = cv2.imread(img_path)
        if image is None:
            raise FileNotFoundError(f'Cannot read: {img_path}')
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        # CPU-side preprocessing (images are NOT pre-processed)
        if self.is_train:
            if random.random() < 0.5: image = apply_clahe(image)
            if random.random() < 0.3: image = apply_sharpening(image)
        # Direct numpy→tensor (no PIL overhead)
        return _np_to_tensor(image), self.labels[idx]


class EarPairwiseDataset(Dataset):
    """Pairwise verification dataset for val/test evaluation."""
    def __init__(self, root_dir, pairs_csv):
        self.root_dir = root_dir
        self.pairs, self.labels = [], []
        with open(os.path.join(root_dir, pairs_csv), 'r') as f:
            for row in csv.reader(f, delimiter='\t'):
                if len(row) >= 3:
                    self.pairs.append((
                        row[0].rsplit('.', 1)[0] + '.webp',
                        row[1].rsplit('.', 1)[0] + '.webp',
                    ))
                    self.labels.append(int(row[2]))
        self._norm = v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])

    def __len__(self): return len(self.pairs)

    def _load(self, relpath):
        full = os.path.join(self.root_dir, relpath)
        raw = cv2.imread(full)
        if raw is None:
            raise FileNotFoundError(f'Cannot read: {full}')
        return self._norm(_np_to_tensor(cv2.cvtColor(raw, cv2.COLOR_BGR2RGB)))

    def __getitem__(self, idx):
        return self._load(self.pairs[idx][0]), self._load(self.pairs[idx][1]), self.labels[idx]


class EarImageDataset(Dataset):
    """Sequestered feature extraction dataset (inference only)."""
    def __init__(self, root_dir):
        self.root_dir = root_dir
        self.images = []
        for dirpath, _, filenames in os.walk(root_dir):
            for fname in sorted(filenames):
                if fname.lower().endswith(('.png', '.jpg', '.jpeg', '.webp')):
                    self.images.append(
                        os.path.relpath(os.path.join(dirpath, fname), root_dir).replace('\\', '/')
                    )
        self._norm = v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])

    def __len__(self): return len(self.images)

    def __getitem__(self, idx):
        raw = cv2.imread(os.path.join(self.root_dir, self.images[idx]))
        if raw is None:
            raise FileNotFoundError(f'Cannot read: {self.images[idx]}')
        image = apply_clahe(cv2.cvtColor(raw, cv2.COLOR_BGR2RGB))
        return self._norm(_np_to_tensor(image)), self.images[idx]

print('Dataset classes defined.')



Dataset classes defined.


In [ ]:
# Cell 5 — Teacher Model (SwinV2-Base) + AdaFace Head

class SwinV2EarModel(nn.Module):
    def __init__(self, embedding_dim=512, freeze_backbone=False):
        super().__init__()
        self.backbone = models.swin_v2_b(weights=models.Swin_V2_B_Weights.DEFAULT)
        self.backbone.head = nn.Identity()
        self.backbone_dim = 1024
        for p in self.backbone.parameters():
            p.requires_grad = not freeze_backbone
        self.projection = nn.Sequential(
            nn.Linear(self.backbone_dim, embedding_dim),
            nn.BatchNorm1d(embedding_dim),
        )

    def forward(self, x, return_norms=False):
        if x.shape[-1] != 256 or x.shape[-2] != 256:
            x = F.interpolate(x, size=(256, 256), mode='bilinear', align_corners=False)
        features = self.backbone(x)
        embeddings = self.projection(features)
        norms = torch.norm(embeddings, p=2, dim=1)
        embeddings = F.normalize(embeddings, p=2, dim=1)
        return (embeddings, norms) if return_norms else embeddings


class AdaFaceHead(nn.Module):
    def __init__(self, in_features, out_features, s=64.0, m=0.25, h=0.333, t_alpha=0.01):
        super().__init__()
        self.s, self.m, self.h, self.t_alpha = s, m, h, t_alpha
        self.out_features = out_features
        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)
        self.register_buffer('batch_mean', torch.ones(1) * 20.0)
        self.register_buffer('batch_std',  torch.ones(1) * 100.0)

    def forward(self, embeddings, norms, labels):
        with torch.no_grad():
            self.batch_mean = (1 - self.t_alpha) * self.batch_mean + self.t_alpha * norms.mean()
            self.batch_std  = (1 - self.t_alpha) * self.batch_std  + self.t_alpha * (norms.std() + 1e-6)
        margin_scaler = torch.clamp(self.h * (norms - self.batch_mean) / (self.batch_std + 1e-3), -1.0, 1.0)
        cos_theta = F.linear(F.normalize(embeddings), F.normalize(self.weight)).clamp(-1+1e-7, 1-1e-7)
        theta = torch.acos(cos_theta)
        one_hot = F.one_hot(labels, self.out_features).float()
        g_add = self.m * (1.0 + margin_scaler)
        target_logits = torch.cos(theta + self.m) - g_add.unsqueeze(1)
        output = (cos_theta * (1 - one_hot) + target_logits * one_hot) * self.s
        return F.cross_entropy(output, labels)

print('Teacher + AdaFace defined.')



Teacher + AdaFace defined.


In [ ]:
# Cell 6 — Evaluation Function & Metrics

def compute_metrics(labels, scores):
    fpr, tpr, _ = roc_curve(labels, scores)
    order = np.argsort(fpr)
    fpr, tpr = fpr[order], tpr[order]
    interp  = interp1d(fpr, tpr, bounds_error=False, fill_value=(0.0, 1.0))
    fnr     = 1 - tpr
    abs_diff = np.abs(fnr - fpr)
    idx = np.argmin(abs_diff)
    if idx > 0 and abs_diff[idx-1] < abs_diff[idx]: idx -= 1
    if idx < len(fpr) - 1:
        x0,x1,y0,y1 = fpr[idx],fpr[idx+1],fnr[idx],fnr[idx+1]
        denom = (x1-x0) + (y0-y1)
        eer = x0 + ((y0-x0)/denom)*(x1-x0) if denom != 0 else (fpr[idx]+fnr[idx])/2
    else:
        eer = (fpr[idx]+fnr[idx])/2
    return {
        'VER@0.1%': float(interp(0.001)),
        'VER@1%'  : float(interp(0.01)),
        'EER'     : float(eer),
        'AUC'     : float(roc_auc_score(labels, scores)),
    }


def evaluate_verification(model, dataloader, device):
    """Works with both plain nn.Module and DataParallel-wrapped models."""
    raw = model.module if hasattr(model, 'module') else model
    raw.eval()
    all_scores, all_labels = [], []
    with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.bfloat16):
        for img1, img2, labels in tqdm(dataloader, desc='Eval', leave=False):
            img1 = img1.to(device, non_blocking=True, memory_format=torch.channels_last)
            img2 = img2.to(device, non_blocking=True, memory_format=torch.channels_last)
            sim  = F.cosine_similarity(raw(img1), raw(img2))
            all_scores.extend(sim.float().cpu().numpy())
            all_labels.extend(labels.numpy())
    return compute_metrics(np.array(all_labels), np.array(all_scores))

print('Evaluation function defined.')



Evaluation function defined.


In [ ]:
# Cell 7 — Build Dataloaders (A100 80 GB Colab Pro+ optimised)

DATA_ROOT   = '/content/UERC26_modified_new/data/public'
CKPT_DIR    = '/content/checkpoints'
PRUNED_CKPT = os.path.join(CKPT_DIR, 'best_pruned_35per_tinyvit.pt')
BATCH_SIZE  = 512
# 6 workers on 8-core Colab Pro+ — leaves 2 cores for main process + pin_memory thread
NUM_WORKERS = 6
EVAL_INTERVAL = 1

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# A100 performance flags
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark        = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32       = True
    torch.set_float32_matmul_precision('high')
    print(f'GPU : {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

# GPU-side augmentation pipeline (applied AFTER moving to device)
gpu_transform = v2.Compose([
    v2.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1),
    v2.RandomGrayscale(p=0.2),
    v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

print('Loading datasets...')
train_dataset = EarIdentificationDataset('train', DATA_ROOT)

# Build test loader
test_csv_path = os.path.join(DATA_ROOT, 'pairs_test.csv')
if os.path.exists(test_csv_path):
    test_pair_dataset = EarPairwiseDataset(DATA_ROOT, 'pairs_test.csv')
    test_loader = DataLoader(
        test_pair_dataset, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True,
        persistent_workers=True, prefetch_factor=4,
    )
    print(f'Test pairs loaded: {len(test_pair_dataset):,}')
else:
    test_loader = None
    print('⚠️  pairs_test.csv not found — test_loader unavailable.')

# Val loader
val_pair_dataset = (
    EarPairwiseDataset(DATA_ROOT, 'pairs_val.csv')
    if os.path.exists(os.path.join(DATA_ROOT, 'pairs_val.csv')) else None
)
val_loader = None
if val_pair_dataset:
    val_loader = DataLoader(
        val_pair_dataset, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True,
        persistent_workers=True, prefetch_factor=4,
    )

# Train loader — NO persistent_workers (avoids stale-worker crash on cell re-runs)
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, drop_last=True,
    persistent_workers=False, prefetch_factor=2,
)

print(f'Train samples : {len(train_dataset):,}')
print(f'Num classes   : {train_dataset.num_classes:,}')
print(f'Batch size    : {BATCH_SIZE}')
print(f'Num workers   : {NUM_WORKERS}')
print(f'Device        : {device}')

GPU : NVIDIA A100-SXM4-80GB
VRAM: 85.1 GB
Loading datasets...
Test pairs loaded: 25,243
Train samples : 179,172
Num classes   : 1,179
Batch size    : 512
Num workers   : 6
Device        : cuda


In [ ]:
# Cell 8 — TinyViT-11M Student Model Definition

class TinyVITStudentModel(nn.Module):
    """
    TinyViT-11M backbone (~21 M params) with a 512-dim AdaFace projection head.
    Input: 224x224 (auto-resized if different).
    Output: L2-normalised 512-dim embedding (+ norms if return_norms=True).
    """
    def __init__(self, embedding_dim=512):
        super().__init__()
        # num_classes=0 returns raw pooled features (448-dim for tiny_vit_11m_224)
        self.backbone = timm.create_model('tiny_vit_11m_224', pretrained=True, num_classes=0)
        self.projection = nn.Sequential(
            nn.Linear(448, embedding_dim),
            nn.BatchNorm1d(embedding_dim),
        )

    def forward(self, x, return_norms=False):
        if x.shape[-1] != 224 or x.shape[-2] != 224:
            x = F.interpolate(x, size=(224, 224), mode='bilinear', align_corners=False)
        feat = self.backbone(x)
        embeddings = self.projection(feat)
        norms = torch.norm(embeddings, p=2, dim=1)      # [B] — quality proxy for AdaFace
        embeddings = F.normalize(embeddings, p=2, dim=1)
        return (embeddings, norms) if return_norms else embeddings


def count_params(m):
    """Count trainable parameters (works with or without DataParallel)."""
    base = m.module if hasattr(m, 'module') else m
    return sum(p.numel() for p in base.parameters() if p.requires_grad)

print('TinyViT-11M student defined.')



TinyViT-11M student defined.


In [ ]:
# Cell 9 — Load Pre-Trained Student + Teacher
#           Runs baseline evaluation ONCE on TEST SET; saved to `baseline_metrics`.

# ── Option A: student checkpoint already in Drive ────────────────
STUDENT_CHECKPOINT_PATH = '/content/drive/MyDrive/2nd_time_UERC26_Output_TinyViT11M_Student_teacher_tinyvit21_superteacher_swinv2/best_student_tinyvit11m.pt'
# ── Option B: upload directly from your machine ──────────────────
# from google.colab import files
# uploaded = files.upload()
# STUDENT_CHECKPOINT_PATH = list(uploaded.keys())[0]
# ─────────────────────────────────────────────────────────────────

# 1. Peek inside checkpoint to get the exact num_classes it was trained with
student_ckpt = torch.load(STUDENT_CHECKPOINT_PATH, map_location=device)

_head_weight = student_ckpt['loss_head_state_dict']['weight']
CKPT_NUM_CLASSES = _head_weight.shape[0]
print(f'Checkpoint num_classes : {CKPT_NUM_CLASSES}')
print(f'Dataset  num_classes   : {train_dataset.num_classes}')
if CKPT_NUM_CLASSES != train_dataset.num_classes:
    print(f'⚠️  Mismatch detected — using checkpoint value ({CKPT_NUM_CLASSES}) for AdaFaceHead.')

# 2. Load student
student_model = TinyVITStudentModel(embedding_dim=512).to(device, memory_format=torch.channels_last)
student_loss_head = AdaFaceHead(in_features=512, out_features=CKPT_NUM_CLASSES).to(device)

# --- FIX: Remove '_orig_mod.' prefix caused by torch.compile ---
raw_state_dict = student_ckpt['model_state_dict']
clean_state_dict = {}
for k, v in raw_state_dict.items():
    name = k.replace('_orig_mod.', '')  # Strip the compilation prefix
    clean_state_dict[name] = v

student_model.load_state_dict(clean_state_dict)
student_loss_head.load_state_dict(student_ckpt['loss_head_state_dict'])
print(f'\n✅ Student loaded | epoch {student_ckpt.get("epoch","N/A")} | val AUC {student_ckpt.get("val_auc",0)*100:.2f}%')

# 3. Load teacher (frozen)
TEACHER_CHECKPOINT_PATH = '/content/drive/MyDrive/unbiased_new_teacher/best_model.pt'
teacher_model = SwinV2EarModel(embedding_dim=512, freeze_backbone=True).to(device, memory_format=torch.channels_last)
teacher_ckpt  = torch.load(TEACHER_CHECKPOINT_PATH, map_location=device)

# Apply same prefix fix for teacher just in case it was also compiled
teacher_raw_sd = teacher_ckpt['model_state_dict']
teacher_clean_sd = {k.replace('_orig_mod.', ''): v for k, v in teacher_raw_sd.items()}

teacher_model.load_state_dict(teacher_clean_sd)
for p in teacher_model.parameters():
    p.requires_grad = False
teacher_model.eval()
print('✅ Teacher loaded and frozen.')

# 4. Param count before pruning
params_before = count_params(student_model)
print(f'Student parameters (before pruning): {params_before:,} ({params_before/1e6:.2f} M)')

# 5. ONE-TIME baseline evaluation on TEST SET — stored in baseline_metrics for Cell 14
if test_loader is not None:
    print('\nRunning baseline evaluation on test set (once)...')
    baseline_metrics = evaluate_verification(student_model, test_loader, device)
    print('\n' + '='*47)
    print('  BASELINE — unpruned TinyViT student (test set)')
    print('='*47)
    for k in ['AUC', 'EER', 'VER@0.1%', 'VER@1%']:
        print(f'  {k:<12}: {baseline_metrics[k]*100:.4f}%')
    print('='*47)
    print('Saved to `baseline_metrics`. Reused automatically in Cell 14.')
else:
    print('⚠️  test_loader unavailable — baseline_metrics empty.')
    baseline_metrics = {}

Checkpoint num_classes : 1179
Dataset  num_classes   : 1179


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/81.5M [00:00<?, ?B/s]


✅ Student loaded | epoch 29 | val AUC 65.28%
Downloading: "https://download.pytorch.org/models/swin_v2_b-781e5279.pth" to /root/.cache/torch/hub/checkpoints/swin_v2_b-781e5279.pth


100%|██████████| 336M/336M [00:01<00:00, 239MB/s]


✅ Teacher loaded and frozen.
Student parameters (before pruning): 10,778,884 (10.78 M)

Running baseline evaluation on test set (once)...


Eval:   0%|          | 0/50 [00:00<?, ?it/s]


  BASELINE — unpruned TinyViT student (test set)
  AUC         : 96.9540%
  EER         : 8.6799%
  VER@0.1%    : 52.0703%
  VER@1%      : 73.1780%
Saved to `baseline_metrics`. Reused automatically in Cell 14.


In [ ]:
# Cell 10 — Verify torch-pruning Installation
import torch_pruning as tp
print(f'torch-pruning version : {tp.__version__}')
print(f'torch version         : {torch.__version__}')
print('Ready for pruning.')



torch-pruning version : 1.6.0
torch version         : 2.10.0+cu128
Ready for pruning.


In [ ]:
# Cell 11 — Configure Pruning Strategy
# ─────────────────────────────────────────────────────────────────
# METHOD  : Taylor 1st-order importance  (|grad z activation|)
#           Scores channels by actual contribution to YOUR KD loss,
#           not just weight magnitude.
#
# TUNE:
#   PRUNING_RATIO    — fraction of channels removed globally
#                      Recommended iteration order: 0.20 → 0.25 → 0.30 → 0.35
#                      Stop when Cell 14 raises ⚠️
#   IMPORTANCE_STEPS — batches used for Taylor scoring
#                      80–100 is good for 180k images on A100
# ─────────────────────────────────────────────────────────────────

PRUNING_RATIO    = 0.45   # <── change this between iterations
IMPORTANCE_STEPS = 100

# Deep-copy student so the original stays intact for baseline comparison
prunable_model = copy.deepcopy(student_model)

# Unwrap before passing to torch-pruning (it cannot trace DataParallel)
base_for_pruning = prunable_model.module if hasattr(prunable_model, 'module') else prunable_model
base_for_pruning.to(device).train()

importance     = tp.importance.TaylorImportance()
ignored_layers = [base_for_pruning.projection]  # never prune the embedding head

# Ignore attention modules to avoid indexing errors on the GPU
for name, m in base_for_pruning.named_modules():
    if 'attn' in name.lower() or 'attention' in type(m).__name__.lower():
        ignored_layers.append(m)

example_inputs = torch.randn(2, 3, 224, 224).to(device)

pruner = tp.pruner.MagnitudePruner(
    base_for_pruning,
    example_inputs,
    importance=importance,
    pruning_ratio=PRUNING_RATIO,
    ignored_layers=ignored_layers,
    global_pruning=False,  # per-layer ratio is safer for hybrid architectures
)

print(f'Pruner configured | ratio={PRUNING_RATIO} | scoring batches={IMPORTANCE_STEPS}')
print(f'Params before pruning: {count_params(base_for_pruning):,}')



/usr/local/lib/python3.12/dist-packages/torch_pruning/dependency/graph.py:390: UserWarning: Unwrapped parameters detected: ['backbone.stages.1.blocks.0.attn.attention_biases', 'backbone.stages.2.blocks.2.attn.attention_biases', 'backbone.stages.1.blocks.1.attn.attention_biases', 'backbone.stages.3.blocks.1.attn.attention_biases', 'backbone.stages.2.blocks.0.attn.attention_biases', 'backbone.stages.2.blocks.1.attn.attention_biases', 'backbone.stages.2.blocks.3.attn.attention_biases', 'backbone.stages.2.blocks.4.attn.attention_biases', 'backbone.stages.2.blocks.5.attn.attention_biases', 'backbone.stages.3.blocks.0.attn.attention_biases'].
 Torch-Pruning will prune the last non-singleton dimension of these parameters. If you wish to change this behavior, please provide an unwrapped_parameters argument.
  warnings.warn(warning_str)


Pruner configured | ratio=0.45 | scoring batches=100
Params before pruning: 10,778,884


In [ ]:
# Cell 12 — Accumulate Taylor Scores → Apply Pruning Masks

mse_loss_fn = nn.MSELoss()
base_for_pruning.train()
teacher_model.eval()   # teacher always frozen

print(f'Accumulating Taylor scores over {IMPORTANCE_STEPS} batches...')
step = 0
for images, labels in tqdm(train_loader, total=IMPORTANCE_STEPS, desc='Taylor scoring'):
    if step >= IMPORTANCE_STEPS:
        break
    images = images.to(device, non_blocking=True, memory_format=torch.channels_last)
    images = gpu_transform(images)

    # Score against KD objective so importance reflects distillation loss,
    # exactly matching the training setup.
    with torch.no_grad():
        teacher_emb = teacher_model(images)
    student_emb = base_for_pruning(images)
    loss = mse_loss_fn(student_emb, teacher_emb)
    loss.backward()
    step += 1

print('✅ Taylor scores accumulated. Applying masks...')
pruner.step()
print('✅ Pruning masks applied.')

# Zero out leftover gradients before re-distillation
base_for_pruning.zero_grad()

params_after = count_params(base_for_pruning)
reduction_pct = (1 - params_after / params_before) * 100
print(f'\nParams before : {params_before:,} ({params_before/1e6:.2f} M)')
print(f'Params after  : {params_after:,} ({params_after/1e6:.2f} M)')
print(f'Reduction     : {reduction_pct:.1f}%')



Accumulating Taylor scores over 100 batches...


Taylor scoring:   0%|          | 0/100 [00:00<?, ?it/s]

✅ Taylor scores accumulated. Applying masks...
✅ Pruning masks applied.

Params before : 10,778,884 (10.78 M)
Params after  : 7,525,990 (7.53 M)
Reduction     : 30.2%


In [ ]:
# Cell 13 — Re-Distillation Fine-Tuning (A100 optimised)
# Recovers accuracy lost from pruning via KD against the frozen teacher.
# ─────────────────────────────────────────────────────────────────
# KEY FIX: Removed GradScaler — it is WRONG for bfloat16.
#          BFloat16 has the same exponent range as float32,
#          so loss scaling is unnecessary and can corrupt gradients.
# ─────────────────────────────────────────────────────────────────

REDIST_EPOCHS = 30
LR_BACKBONE   = 5e-5
LR_HEAD       = 2e-4
ALPHA_KD      = 0.70

# Fresh AdaFace head for the pruned model
pruned_loss_head = AdaFaceHead(in_features=512, out_features=train_dataset.num_classes).to(device)

# Fused AdamW: single CUDA kernel for param update — ~15% faster on A100
optimizer_p = torch.optim.AdamW([
    {'params': base_for_pruning.backbone.parameters(),   'lr': LR_BACKBONE},
    {'params': base_for_pruning.projection.parameters(), 'lr': LR_HEAD},
    {'params': pruned_loss_head.parameters(),            'lr': LR_HEAD},
], weight_decay=1e-4, fused=True)

scheduler_p = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_p, T_max=REDIST_EPOCHS, eta_min=1e-6)
mse_loss_fn = nn.MSELoss()

# torch.compile for A100 — fuses ops, reduces kernel launches
# Applied AFTER pruning so the compiled graph matches the pruned architecture
compiled_student = torch.compile(base_for_pruning)

best_pruned_ver = 0.0

print(f'Re-distillation: {REDIST_EPOCHS} epochs | alpha_kd={ALPHA_KD} | LR_backbone={LR_BACKBONE}')
print(f'Params: {params_after:,} (was {params_before:,}, -{(1-params_after/params_before)*100:.1f}%)')

for epoch in range(1, REDIST_EPOCHS + 1):
    compiled_student.train()
    pruned_loss_head.train()
    total_loss, num_batches = 0.0, 0

    pbar = tqdm(train_loader, desc=f'Re-Dist {epoch}/{REDIST_EPOCHS}', leave=False)
    for images, labels in pbar:
        images = images.to(device, non_blocking=True, memory_format=torch.channels_last)
        labels = labels.to(device, non_blocking=True)
        images = gpu_transform(images)

        optimizer_p.zero_grad(set_to_none=True)

        with torch.amp.autocast('cuda', dtype=torch.bfloat16):
            with torch.no_grad():
                teacher_emb = teacher_model(images)
            student_emb, student_norms = compiled_student(images, return_norms=True)
            loss_hard = pruned_loss_head(student_emb, student_norms, labels)
            loss_soft = mse_loss_fn(student_emb, teacher_emb)
            loss      = (1.0 - ALPHA_KD) * loss_hard + ALPHA_KD * loss_soft

        # No GradScaler needed for bfloat16
        loss.backward()
        torch.nn.utils.clip_grad_norm_(base_for_pruning.parameters(), 1.0)
        torch.nn.utils.clip_grad_norm_(pruned_loss_head.parameters(), 1.0)
        optimizer_p.step()

        total_loss  += loss.item()
        num_batches += 1
        if num_batches % 10 == 0:  # update every 10 steps to reduce CPU overhead
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    avg_loss = total_loss / max(num_batches, 1)
    print(f'Epoch {epoch}/{REDIST_EPOCHS} | Loss: {avg_loss:.4f} | LR: {scheduler_p.get_last_lr()[0]:.2e}')


    # Evaluate and save best checkpoint every epoch
    if val_loader is not None:
        metrics = evaluate_verification(base_for_pruning, val_loader, device)
        print(
            f'  VAL | AUC: {metrics["AUC"]*100:.2f}% | '
            f'VER@0.1%: {metrics["VER@0.1%"]*100:.2f}% | '
            f'EER: {metrics["EER"]*100:.2f}%'
        )
        if metrics['VER@0.1%'] > best_pruned_ver:
            best_pruned_ver = metrics['VER@0.1%']
            torch.save({
                'epoch':                epoch,
                'model_state_dict':     base_for_pruning.state_dict(),
                'loss_head_state_dict': pruned_loss_head.state_dict(),
                'val_ver':              best_pruned_ver,
                'params':               params_after,
                'pruning_ratio':        PRUNING_RATIO,
                'embedding_dim':        512,
                'backbone':             'tiny_vit_21m_pruned',
            }, PRUNED_CKPT)
            print(f'  ★ NEW BEST SAVED (VER@0.1%: {best_pruned_ver*100:.2f}%)')

    scheduler_p.step()

print(f'\nRe-distillation done! Best VER@0.1%: {best_pruned_ver*100:.2f}%')
print(f'Params: {params_after:,} ({params_after/1e6:.2f} M)  [was {params_before/1e6:.2f} M, -{(1-params_after/params_before)*100:.1f}%]')




Re-distillation: 30 epochs | alpha_kd=0.7 | LR_backbone=5e-05
Params: 7,525,990 (was 10,778,884, -30.2%)


Re-Dist 1/30:   0%|          | 0/349 [00:00<?, ?it/s]

Epoch 1/30 | Loss: 10.5362 | LR: 5.00e-05


Eval:   0%|          | 0/357 [00:00<?, ?it/s]

  VAL | AUC: 95.40% | VER@0.1%: 38.54% | EER: 11.20%
  ★ NEW BEST SAVED (VER@0.1%: 38.54%)


...

  VAL | AUC: 97.44% | VER@0.1%: 64.09% | EER: 7.47%


Re-Dist 30/30:   0%|          | 0/349 [00:00<?, ?it/s]

Epoch 30/30 | Loss: 5.0342 | LR: 1.13e-06


Eval:   0%|          | 0/357 [00:00<?, ?it/s]

  VAL | AUC: 97.45% | VER@0.1%: 63.78% | EER: 7.44%

Re-distillation done! Best VER@0.1%: 64.27%
Params: 7,525,990 (7.53 M)  [was 10.78 M, -30.2%]


In [ ]:
# Cell 14 — Compare Pruned vs Baseline (test set)
# Baseline comes from the ONE-TIME evaluation in Cell 9.
# ⚠️  fires when: AUC drop > 3%  OR  VER@0.1% drop > 5%

if not test_loader or not baseline_metrics:
    print('⚠️  No test loader or baseline metrics. Skipping comparison.')
else:
    # Load best pruned weights for a clean comparison
    if os.path.exists(PRUNED_CKPT):
        base_for_pruning.load_state_dict(
            torch.load(PRUNED_CKPT, map_location=device)['model_state_dict']
        )
    pruned_metrics = evaluate_verification(base_for_pruning, test_loader, device)

    W = 58
    print('\n' + '='*W)
    print(f'  {"METRIC":<14}{"BASELINE":>10}{"PRUNED":>10}{"DELTA":>9}  STATUS')
    print('='*W)
    floors = {'AUC': -3.0, 'VER@0.1%': -5.0, 'EER': None, 'VER@1%': None}
    all_ok = True
    for k in ['AUC', 'EER', 'VER@0.1%', 'VER@1%']:
        b, p  = baseline_metrics[k]*100, pruned_metrics[k]*100
        delta = p - b
        floor = floors[k]
        flag  = '⚠️  OVER' if (floor is not None and delta < floor) else '✅  OK'
        if '⚠️' in flag: all_ok = False
        print(f'  {k:<14}{b:>9.3f}%{p:>9.3f}%{delta:>+8.3f}%  {flag}')
    print('='*W)
    print(f'  Params: {params_before:,} → {params_after:,}  (-{(1-params_after/params_before)*100:.1f}%)')
    print('='*W)
    if all_ok:
        print('\n✅ All metrics within tolerance.')
        print('   → Want more compression? Raise PRUNING_RATIO in Cell 11 and re-run Cells 11–16.')
    else:
        print('\n⚠️  Metric floor breached.')
        print('   → Lower PRUNING_RATIO in Cell 11 and re-run Cells 11–16.')

Eval:   0%|          | 0/50 [00:00<?, ?it/s]


  METRIC          BASELINE    PRUNED    DELTA  STATUS
  AUC              96.954%   96.884%  -0.070%  ✅  OK
  EER               8.680%    8.993%  +0.313%  ✅  OK
  VER@0.1%         52.070%   50.646%  -1.424%  ✅  OK
  VER@1%           73.178%   72.107%  -1.071%  ✅  OK
  Params: 10,778,884 → 7,525,990  (-30.2%)

✅ All metrics within tolerance.
   → Want more compression? Raise PRUNING_RATIO in Cell 11 and re-run Cells 11–16.


In [ ]:
# Cell 15 — Full Test-Set Evaluation on Best Pruned Model
# Run after every pruning iteration to verify test-set floors.

TEST_CSV      = 'pairs_test.csv'
test_csv_path = os.path.join(DATA_ROOT, TEST_CSV)

if not os.path.exists(PRUNED_CKPT):
    print(f'❌ No pruned checkpoint at {PRUNED_CKPT}. Run Cells 11→13 first.')
elif not os.path.exists(test_csv_path):
    print(f'❌ Test CSV not found at {test_csv_path}.')
else:
    pruned_ck = torch.load(PRUNED_CKPT, map_location=device)
    base_for_pruning.load_state_dict(pruned_ck['model_state_dict'])

    test_pair_dataset = EarPairwiseDataset(DATA_ROOT, TEST_CSV)
    test_loader_p = DataLoader(
        test_pair_dataset, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True,
        persistent_workers=True, prefetch_factor=2,
    )
    test_metrics = evaluate_verification(base_for_pruning, test_loader_p, device)

    print('\n' + '='*47)
    print('  PRUNED MODEL — FULL TEST SET')
    print('='*47)
    print(f'  Backbone    : TinyViT-11M (pruned)')
    print(f'  Parameters  : {pruned_ck["params"]:,}  ({pruned_ck["params"]/1e6:.2f} M)')
    print(f'  Was         : {params_before:,}  ({params_before/1e6:.2f} M)')
    print(f'  Reduction   : {(1-pruned_ck["params"]/params_before)*100:.1f}%')
    print(f'  Prune ratio : {pruned_ck["pruning_ratio"]}')
    print(f'  Best epoch  : {pruned_ck["epoch"]}')
    print('  ---')
    print(f'  VER@0.1%    : {test_metrics["VER@0.1%"]*100:.4f}%')
    print(f'  VER@1%      : {test_metrics["VER@1%"]*100:.4f}%')
    print(f'  EER         : {test_metrics["EER"]*100:.4f}%')
    print(f'  AUC         : {test_metrics["AUC"]*100:.4f}%')
    print('='*47)



Eval:   0%|          | 0/50 [00:00<?, ?it/s]


  PRUNED MODEL — FULL TEST SET
  Backbone    : TinyViT-11M (pruned)
  Parameters  : 7,525,990  (7.53 M)
  Was         : 10,778,884  (10.78 M)
  Reduction   : 30.2%
  Prune ratio : 0.45
  Best epoch  : 26
  ---
  VER@0.1%    : 50.6465%
  VER@1%      : 72.1074%
  EER         : 8.9926%
  AUC         : 96.8844%


In [ ]:
import shutil

DRIVE_PRUNED_DIR = '/content/drive/MyDrive/UERC26_Output_pruned_tinyvit_11m'
os.makedirs(DRIVE_PRUNED_DIR, exist_ok=True)

if os.path.exists(PRUNED_CKPT):
    shutil.copy(PRUNED_CKPT, DRIVE_PRUNED_DIR)
    ck = torch.load(PRUNED_CKPT, map_location='cpu')
    print(f'✅ Saved to: {DRIVE_PRUNED_DIR}')
    print(f'   Val VER@0.1%: {ck["val_ver"]*100:.4f}%')
    print(f'   Parameters : {ck["params"]:,}  ({ck["params"]/1e6:.2f} M)')
    print(f'   Prune ratio: {ck["pruning_ratio"]}')
    print(f'   Epoch      : {ck["epoch"]}')
else:
    print('❌ No pruned checkpoint found. Run Cells 11→13 first.')

print('\n' + '─'*55)
print('📌 NEXT STEPS:')
print('   • Cells 17–18 use base_for_pruning + PRUNED_CKPT.')
print('     No path changes needed — wired in automatically.')
print('   • To prune further: change PRUNING_RATIO in Cell 11,')
print('     then re-run Cells 11 → 12 → 13 → 14 → 15 → 16.')
print('─'*55)



✅ Saved to: /content/drive/MyDrive/UERC26_Output_pruned_tinyvit_11m
   Val VER@0.1%: 64.2747%
   Parameters : 7,525,990  (7.53 M)
   Prune ratio: 0.45
   Epoch      : 26

───────────────────────────────────────────────────────
📌 NEXT STEPS:
   • Cells 17–18 use base_for_pruning + PRUNED_CKPT.
     No path changes needed — wired in automatically.
   • To prune further: change PRUNING_RATIO in Cell 11,
     then re-run Cells 11 → 12 → 13 → 14 → 15 → 16.
───────────────────────────────────────────────────────


In [ ]:
# Cell 17 — Extract Sequestered Features & Save to Drive (Submission)
# Uses the best pruned model from PRUNED_CKPT.

import shutil

def extract_embeddings(model, dataloader, device):
    raw = model.module if hasattr(model, 'module') else model
    raw.eval()
    features = {}
    with torch.no_grad():
        for images, img_names in tqdm(dataloader, desc='Extracting', leave=False):
            emb = raw(images.to(device, memory_format=torch.channels_last)).cpu().numpy()
            for name, feat in zip(img_names, emb):
                features[name] = feat
    return features

if not os.path.exists(PRUNED_CKPT):
    raise FileNotFoundError(f'Pruned checkpoint not found: {PRUNED_CKPT}. Run Cells 11–13 first.')

pruned_ck = torch.load(PRUNED_CKPT, map_location=device)
base_for_pruning.load_state_dict(pruned_ck['model_state_dict'])
base_for_pruning.eval()
print(f'✅ Loaded pruned model | params: {pruned_ck["params"]/1e6:.2f} M | VER@0.1%: {pruned_ck["val_ver"]*100:.2f}%')

seq_root      = '/content/UERC26_modified_new/data/sequestered_anonymized'
local_out_dir = '/content/checkpoints/tinyvit_pruned_solution'
DRIVE_OUT_DIR = '/content/drive/MyDrive/UERC26_11M_35%_Pruned_Output_tinyvit'
os.makedirs(local_out_dir, exist_ok=True)
os.makedirs(DRIVE_OUT_DIR, exist_ok=True)

if not os.path.exists(seq_root):
    print(f'❌ Sequestered data not found at {seq_root}.')
    print('   Upload the sequestered dataset and update seq_root.')
else:
    print(f'Extracting features from {seq_root}...')
    seq_dataset = EarImageDataset(seq_root)
    seq_loader  = DataLoader(
        seq_dataset, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True, prefetch_factor=2,
    )
    seq_features = extract_embeddings(base_for_pruning, seq_loader, device)
    npy_path = os.path.join(local_out_dir, 'sequestered_features.npy')
    np.save(npy_path, seq_features)
    shutil.copy(npy_path, DRIVE_OUT_DIR)
    shutil.copy(PRUNED_CKPT, DRIVE_OUT_DIR)
    print(f'✅ {len(seq_features):,} embeddings saved to Drive: {DRIVE_OUT_DIR}')
    print(f'✅ Pruned checkpoint copied to Drive.')



✅ Loaded pruned model | params: 7.53 M | VER@0.1%: 64.27%
Extracting features from /content/UERC26_modified_new/data/sequestered_anonymized...


Extracting:   0%|          | 0/4 [00:00<?, ?it/s]

✅ 1,670 embeddings saved to Drive: /content/drive/MyDrive/UERC26_11M_35%_Pruned_Output_tinyvit
✅ Pruned checkpoint copied to Drive.


In [ ]:
# Cell 18 — Final Evaluation on Test Set (Pruned Model)

TEST_CSV      = 'pairs_test.csv'   # change to pairs_val.csv if test CSV unavailable
test_csv_path = os.path.join(DATA_ROOT, TEST_CSV)

if not os.path.exists(PRUNED_CKPT):
    print(f'❌ Pruned checkpoint not found: {PRUNED_CKPT}')
elif not os.path.exists(test_csv_path):
    print(f'❌ {TEST_CSV} not found at {DATA_ROOT}')
else:
    pruned_ck = torch.load(PRUNED_CKPT, map_location=device)
    base_for_pruning.load_state_dict(pruned_ck['model_state_dict'])

    test_pair_dataset = EarPairwiseDataset(DATA_ROOT, TEST_CSV)
    test_loader_final = DataLoader(
        test_pair_dataset, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True,
        persistent_workers=True, prefetch_factor=2,
    )
    final_metrics = evaluate_verification(base_for_pruning, test_loader_final, device)

    print('\n' + '='*42)
    print('  FINAL PRUNED MODEL — TEST SET RESULTS')
    print('='*42)
    print(f'  Backbone        : TinyViT-11M (pruned)')
    print(f'  Parameters      : {pruned_ck["params"]/1e6:.2f} M  (was {params_before/1e6:.2f} M)')
    print(f'  Param reduction : {(1-pruned_ck["params"]/params_before)*100:.1f}%')
    print(f'  Best epoch      : {pruned_ck["epoch"]}')
    print('  ---')
    print(f'  VER @ 0.1% FAR  : {final_metrics["VER@0.1%"]*100:.4f}%')
    print(f'  VER @ 1.0% FAR  : {final_metrics["VER@1%"]*100:.4f}%')
    print(f'  Equal Error Rate: {final_metrics["EER"]*100:.4f}%')
    print(f'  ROC AUC         : {final_metrics["AUC"]*100:.4f}%')
    print('='*42)



Eval:   0%|          | 0/50 [00:00<?, ?it/s]


  FINAL PRUNED MODEL — TEST SET RESULTS
  Backbone        : TinyViT-11M (pruned)
  Parameters      : 7.53 M  (was 10.78 M)
  Param reduction : 30.2%
  Best epoch      : 26
  ---
  VER @ 0.1% FAR  : 50.6465%
  VER @ 1.0% FAR  : 72.1074%
  Equal Error Rate: 8.9926%
  ROC AUC         : 96.8844%


In [ ]:
import os
os.kill(os.getpid(), 9)